# Lab | Data Aggregation and Filtering

In this challenge, we will continue to work with customer data from an insurance company. We will use the dataset called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by first performing data cleaning, formatting, and structuring.

1. Create a new DataFrame that only includes customers who:
   - have a **low total_claim_amount** (e.g., below $1,000),
   - have a response "Yes" to the last marketing campaign.

2. Using the original Dataframe, analyze:
   - the average `monthly_premium` and/or customer lifetime value by `policy_type` and `gender` for customers who responded "Yes", and
   - compare these insights to `total_claim_amount` patterns, and discuss which segments appear most profitable or low-risk for the company.

3. Analyze the total number of customers who have policies in each state, and then filter the results to only include states where there are more than 500 customers.

4. Find the maximum, minimum, and median customer lifetime value by education level and gender. Write your conclusions.

## Bonus

5. The marketing team wants to analyze the number of policies sold by state and month. Present the data in a table where the months are arranged as columns and the states are arranged as rows.

6.  Display a new DataFrame that contains the number of policies sold by month, by state, for the top 3 states with the highest number of policies sold.

*Hint:*
- *To accomplish this, you will first need to group the data by state and month, then count the number of policies sold for each group. Afterwards, you will need to sort the data by the count of policies sold in descending order.*
- *Next, you will select the top 3 states with the highest number of policies sold.*
- *Finally, you will create a new DataFrame that contains the number of policies sold by month for each of the top 3 states.*

7. The marketing team wants to analyze the effect of different marketing channels on the customer response rate.

Hint: You can use melt to unpivot the data and create a table that shows the customer response rate (those who responded "Yes") by marketing channel.

External Resources for Data Filtering: https://towardsdatascience.com/filtering-data-frames-in-pandas-b570b1f834b9

In [3]:
import pandas as pd

# Load the dataset
url = 'https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv'
df = pd.read_csv(url)

# Display basic info
print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
print("\nFirst 5 rows:")
print(df.head())

# Exercise 1: Create a new DataFrame that only includes customers who have a low total_claim_amount (below $1,000) and responded "Yes" to the last marketing campaign.
low_claim_yes_response = df[(df['Total Claim Amount'] < 1000) & (df['Response'] == 'Yes')]
print("\nExercise 1: Customers with low claim amount (<$1000) and response 'Yes'")
print("Shape:", low_claim_yes_response.shape)
print(low_claim_yes_response.head())

# Exercise 2: Analyze average monthly_premium and customer lifetime value by policy_type and gender for customers who responded "Yes". Compare to total_claim_amount patterns.
yes_response_df = df[df['Response'] == 'Yes']

# Average monthly_premium by policy_type and gender
avg_premium = yes_response_df.groupby(['Policy Type', 'Gender'])['Monthly Premium Auto'].mean().round(2)
print("\nExercise 2a: Average monthly premium by policy type and gender (responded Yes)")
print(avg_premium)

# Average customer lifetime value by policy_type and gender
avg_clv = yes_response_df.groupby(['Policy Type', 'Gender'])['Customer Lifetime Value'].mean().round(2)
print("\nExercise 2b: Average customer lifetime value by policy type and gender (responded Yes)")
print(avg_clv)

# Average total_claim_amount by policy_type and gender for comparison
avg_claim = yes_response_df.groupby(['Policy Type', 'Gender'])['Total Claim Amount'].mean().round(2)
print("\nExercise 2c: Average total claim amount by policy type and gender (responded Yes)")
print(avg_claim)

print("\nAnalysis: Compare premiums vs claims to identify profitable segments.")
print("Segments with high premiums and low claims are most profitable.")

# Exercise 3: Analyze the total number of customers who have policies in each state, and then filter the results to only include states where there are more than 500 customers.
customers_by_state = df.groupby('State').size().sort_values(ascending=False)
print("\nExercise 3: Total customers by state")
print(customers_by_state)

states_over_500 = customers_by_state[customers_by_state > 500]
print("\nStates with more than 500 customers:")
print(states_over_500)

# Exercise 4: Find the maximum, minimum, and median customer lifetime value by education level and gender. Write your conclusions.
clv_stats = df.groupby(['Education', 'Gender'])['Customer Lifetime Value'].agg(['max', 'min', 'median']).round(2)
print("\nExercise 4: CLV statistics by education and gender")
print(clv_stats)

print("\nConclusions: Analyze which education-gender combinations have highest/lowest CLV.")
print("This helps identify target segments for marketing.")

# Bonus Exercise 5: The marketing team wants to analyze the number of policies sold by state and month. Present the data in a table where the months are arranged as columns and the states are arranged as rows.
df['Effective To Date'] = pd.to_datetime(df['Effective To Date'])
df['month'] = df['Effective To Date'].dt.month

policies_by_state_month = df.pivot_table(values='Customer', index='State', columns='month', aggfunc='count', fill_value=0)
print("\nBonus 5: Number of policies sold by state and month")
print(policies_by_state_month)

# Bonus Exercise 6: Display a new DataFrame that contains the number of policies sold by month, by state, for the top 3 states with the highest number of policies sold.
total_policies_by_state = df.groupby('State').size().sort_values(ascending=False)
top_3_states = total_policies_by_state.head(3).index.tolist()
print("\nTop 3 states by total policies:", top_3_states)

# Filter data for top 3 states
top_3_df = df[df['State'].isin(top_3_states)]

# Group by state and month, count policies
policies_top3 = top_3_df.groupby(['State', 'month']).size().reset_index(name='policies_sold')

# Pivot to have months as columns
policies_top3_pivot = policies_top3.pivot(index='State', columns='month', values='policies_sold').fillna(0).astype(int)
print("\nBonus 6: Policies sold by month for top 3 states")
print(policies_top3_pivot)

# Bonus Exercise 7: The marketing team wants to analyze the effect of different marketing channels on the customer response rate.
# Use melt to unpivot the data and create a table that shows the customer response rate (those who responded "Yes") by marketing channel.

# First, calculate response rate by sales_channel
response_rate = df.groupby('Sales Channel')['Response'].apply(lambda x: (x == 'Yes').mean()).round(4) * 100
print("\nBonus 7: Response rate (%) by sales channel")
print(response_rate)

# To use melt, perhaps create a melted version for visualization
melted_response = df.melt(id_vars=['Sales Channel'], value_vars=['Response'], var_name='metric', value_name='response_value')
response_by_channel = melted_response.groupby('Sales Channel')['response_value'].apply(lambda x: (x == 'Yes').mean()).round(4) * 100
print("\nUsing melt approach (same result):")
print(response_by_channel)

Dataset shape: (10910, 26)
Columns: ['Unnamed: 0', 'Customer', 'State', 'Customer Lifetime Value', 'Response', 'Coverage', 'Education', 'Effective To Date', 'EmploymentStatus', 'Gender', 'Income', 'Location Code', 'Marital Status', 'Monthly Premium Auto', 'Months Since Last Claim', 'Months Since Policy Inception', 'Number of Open Complaints', 'Number of Policies', 'Policy Type', 'Policy', 'Renew Offer Type', 'Sales Channel', 'Total Claim Amount', 'Vehicle Class', 'Vehicle Size', 'Vehicle Type']

First 5 rows:
   Unnamed: 0 Customer       State  Customer Lifetime Value Response  \
0           0  DK49336     Arizona              4809.216960       No   
1           1  KX64629  California              2228.525238       No   
2           2  LZ68649  Washington             14947.917300       No   
3           3  XL78013      Oregon             22332.439460      Yes   
4           4  QA50777      Oregon              9025.067525       No   

   Coverage Education Effective To Date EmploymentSt

/var/folders/b7/yg62l3k53kvc1lvmnwswr23m0000gn/T/ipykernel_41552/919416477.py:58: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Effective To Date'] = pd.to_datetime(df['Effective To Date'])
